# 07. Hedonikus Ármodell (OLS Regresszió)

**Cél:** az áralakító paraméterek (méret, szobaszám, állapot, vasúttávolság) izolált, ceteris paribus értékének matematikai meghatározása.

---

### 📖 A hedonikus árelmélet alapjai (Rosen, 1974)
A lakás hasznosságot hordozó tulajdonságok „csomagja": a vevő négyzetmétert, szobákat, infrastrukturális adottságokat és távolságokat vásárol. A hedonikus regresszió ezen tulajdonságok rejtett (implicit) árát egyenként, a többi tényező változatlansága mellett becsli.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elérhető eladó minták: {len(elado)} db.")

### 1. Az OLS Modell Eredményei és az Együtthatók

**📌 A modell specifikációja:**
Függő változó a fajlagos négyzetméterár természetes logaritmusa — így az együtthatók közvetlenül százalékos prémiumként/diszkontként értelmezhetők.

**🔍 Értelmezés:**
- Érvényesnek a 95%-os szignifikanciaszintet (p < 0,05) elérő változókat tekintjük.
- A determinációs együttható (R²) mutatja, hogy a változók az árvariancia hány százalékát magyarázzák; a megmagyarázatlan rész az egyedi, nem számszerűsíthető jellemzőket és a (08. fejezetben igazolandó) térbeli autokorrelációt tartalmazza.

In [ ]:
# Egységesített ökonometriai kontrollváltozók
elado['log_tavolsag_vasut_m'] = np.log(elado['tavolsag_vasut_m'].replace(0, 1))
elado['emelet_szam'] = elado['emelet_szam'].fillna(elado['emelet_szam'].median())
elado['allapot_kod'] = elado['allapot_kod'].fillna(elado['allapot_kod'].median())

features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel',
    'has_lift', 'allapot_kod', 'has_erkely', 'emelet_szam',
    'tavolsag_metro_halozati_m',
    'log_tavolsag_vasut_m',            # PÁLYATEST LÉGVONAL (zaj externália, rugalmasság)
    'tavolsag_vasut_halozati_m'        # ÁLLOMÁS HÁLÓZAT (TOD elérhetőség)
]
# A hálózati változók csak a garantáltan pontos almintán definiáltak:
# a modell erre az almintára szűkül (strukturális szelekció, nem adathiány).
df_reg = elado.dropna(subset=['log_nm_ar'] + features).copy()
print(f"Becslési minta: {len(df_reg)} db a {len(elado)} eladóból — a szűkülés oka: "
      "a hálózati távolságok csak a garantált pontos almintán definiáltak.")

X = sm.add_constant(df_reg[features])
y = df_reg['log_nm_ar']

model_ols = sm.OLS(y, X).fit(cov_type='HC1')   # HC1 robusztus standard hibák

dummy_vars = {'is_panel', 'has_lift', 'has_erkely'}

def implicit_hatas(v):
    b = model_ols.params[v]
    if v in dummy_vars:
        return f"{(np.exp(b)-1)*100:+.1f}%"
    if v == 'log_tavolsag_vasut_m':
        return f"rugalmasság: {b:+.3f}"
    return f"{b:+.5f} / egység"

def sig_label(p):
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else 'ns'))

r2 = model_ols.rsquared
r2_adj = model_ols.rsquared_adj
n_obs = int(model_ols.nobs)
zaj_coef = model_ols.params.get('log_tavolsag_vasut_m', 0)
tod_coef = model_ols.params.get('tavolsag_vasut_halozati_m', 0)
panel_coef = model_ols.params.get('is_panel', 0)
zaj_p = model_ols.pvalues.get('log_tavolsag_vasut_m', 1)
tod_p = model_ols.pvalues.get('tavolsag_vasut_halozati_m', 1)

kpi_cards = [
    ("Modell R²", f"{r2:.3f}", f"Adj. R²: {r2_adj:.3f} (HC1 SE)", "#1e3a8a"),
    ("Vasúti zaj rugalmasság", f"{zaj_coef:+.3f}", f"p = {zaj_p:.4f} {sig_label(zaj_p)}", "#ef4444"),
    ("Állomás TOD hatás (/100m)", f"{tod_coef*100:+.3f}", f"p = {tod_p:.4f} {sig_label(tod_p)}", "#f59e0b"),
    ("Panel diszkont", f"{(np.exp(panel_coef)-1)*100:+.1f}%", "Ceteris paribus (dummy)", "#dc2626"),
    ("Mintaelemszám (N)", f"{n_obs:,} db".replace(',', ' '), "Garantált pontos eladó alminta", "#059669"),
    ("Állapot felár / kategória", f"{(np.exp(model_ols.params['allapot_kod'])-1)*100:+.1f}%", "Műszaki prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

valtozo_magyarazat = {
    'const': 'Tengelymetszet (Konstans)',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám összesen',
    'is_panel': 'Panelszerkezet dummy (1=panel)',
    'has_lift': 'Lift dummy (1=van lift)',
    'allapot_kod': 'Műszaki állapot index (1-6 skála)',
    'has_erkely': 'Erkély dummy (1=van erkély)',
    'emelet_szam': 'Emelet szintszám',
    'tavolsag_metro_halozati_m': 'Metróállomás hálózati távolság (m)',
    'log_tavolsag_vasut_m': 'Vasúti pálya log légvonal (ln m) — zaj externália',
    'tavolsag_vasut_halozati_m': 'Vasútállomás hálózati távolság (m) — TOD'
}

res_df = pd.DataFrame({
    'Változó': [valtozo_magyarazat.get(c, c) for c in model_ols.params.index],
    'Együttható (β)': model_ols.params.values,
    'Std. Hiba (HC1)': model_ols.bse.values,
    't-érték': model_ols.tvalues.values,
    'p-érték': model_ols.pvalues.values,
    'Szign.': [sig_label(p) for p in model_ols.pvalues.values],
    'Implicit Hatás': [implicit_hatas(v) if v in valtozo_magyarazat else '' for v in model_ols.params.index]
})
res_table_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + res_df.round(4).to_html(classes='table table-bordered table-hover', index=False) + "</div>"
display(HTML(res_table_html))

from statsmodels.stats.diagnostic import het_breuschpagan
_, bp_p, _, _ = het_breuschpagan(model_ols.resid, model_ols.model.exog)
print(f"Breusch–Pagan teszt: p = {bp_p:.4f} "
      + ("— heteroszkedaszticitás jelen lehet (a HC1 SE-k ezt kezelik)." if bp_p < 0.05 else "— nincs kimutatható heteroszkedaszticitás."))
try:
    reset = sm.stats.diagnostic.linear_reset(model_ols, power=2, use_f=True)
    print(f"RESET teszt (F): {reset.statistic:.2f}, p = {reset.pvalue:.4f} "
          + ("— specifikációs hiba gyanúja." if reset.pvalue < 0.05 else "— nincs specifikációs hiba jele."))
except Exception as e:
    print(f"RESET nem futtatható: {e}")


### 2. A Becsült Együtthatók Részletes Értelmezése

**📌 Mit jelentenek a számok a valóságban?**
1. **Állapot-prémium (+7.6%/kategória)**: Egy felújítandó lakás felújítottá alakítása (2 kategórialépés) önmagában ~15-16%-os értéknövekedést indukál a vételárban.
2. **Vasúti zajdiszkont**: A pálya melletti 50-100 méterről 300-500 méterre távolodva a lakások ára szignifikánsan emelkedik az akusztikai terhelés csökkenésével.
3. **Paneldiszkont (-10.7%)**: Minden más tényezőt kiszűrve a panel szerkezet önmagában ~10.7%-kal olcsóbbá teszi az ingatlant a téglához viszonyítva.

In [ ]:
df_reg['y_pred'] = model_ols.fittedvalues
df_reg['resid'] = model_ols.resid

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Tényleges vs. Becsült ln(Ár/m²)', 'Maradványértékek Eloszlása', 'Reziduálisok vs. Becsült Értékek')
)

# 1. Tényleges vs Becsült
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['log_nm_ar'], mode='markers', marker=dict(color='#2563eb', opacity=0.6), name='Adatpontok'),
    row=1, col=1
)
min_val = min(df_reg['y_pred'].min(), df_reg['log_nm_ar'].min())
max_val = max(df_reg['y_pred'].max(), df_reg['log_nm_ar'].max())
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', line=dict(color='red', dash='dash'), name='y=x (ideális)'),
    row=1, col=1
)

# 2. Reziduális hisztogram
fig1.add_trace(
    go.Histogram(x=df_reg['resid'], nbinsx=30, marker_color='#059669', opacity=0.8, name='Maradványok'),
    row=1, col=2
)

# 3. Residuals vs Fitted
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['resid'], mode='markers', marker=dict(color='#d97706', opacity=0.6), name='Hiba'),
    row=1, col=3
)
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[0, 0], mode='lines', line=dict(color='black', dash='dash'), showlegend=False),
    row=1, col=3
)

fig1.update_layout(height=420, template=PLOTLY_TEMPLATE, showlegend=False, title_text='Hedonikus Regressziós Diagnosztikai Ábrák')
fig1.show()

# VIF számítás
vif_data = pd.DataFrame()
vif_data["Változó"] = [valtozo_magyarazat.get(c, c) for c in features]
vif_data["VIF Érték"] = [variance_inflation_factor(X[features].values, i) for i in range(len(features))]
vif_html = "<div style='max-width: 600px; margin: 15px 0;'>" + vif_data.round(2).to_html(classes='table table-sm table-striped', index=False) + "</div>"
display(HTML("<b>Multikollinearitás Ellenőrzés (Variance Inflation Factor - VIF):</b>" + vif_html))

### 3. Együtthatók és Konfidencia Intervallumok (Forest Plot)

**📌 Hogyan olvassuk a grafikont?**
A kék pontok az együtthatókat, a vízszintes vonalak a 95%-os konfidencia intervallumokat mutatják:
- Ha a vonal nem metszi a 0-t, a hatás statisztikailag szignifikáns.
- Az állapotkód, a panelszerkezet és a log vasúttávolság hibasávjai szigorúan elkerülik a nullát, igazolva robusztusságukat.

In [ ]:
# 3 modell lépcsőzetes ökonometriai becslése (Hierarchikus specifikáció)
# Modell 1: Fizikai ingatlanstruktúra kontrolljai (Alapterület, Szobaszám, Panel, Lift, Állapot, Erkély, Emelet)
m1_feats = ['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift', 'allapot_kod', 'has_erkely', 'emelet_szam']
# Modell 2: + Közlekedési elérhetőség (Metró + Vasútállomás gyalogos elérhetőség)
m2_feats = m1_feats + ['tavolsag_metro_halozati_m', 'tavolsag_vasut_halozati_m']
# Modell 3: + Környezeti zajexternália (Kettős TOD modell vasúti zajjal)
m3_feats = m2_feats + ['log_tavolsag_vasut_m']

m1 = sm.OLS(y, sm.add_constant(df_reg[m1_feats])).fit(cov_type='HC1')
m2 = sm.OLS(y, sm.add_constant(df_reg[m2_feats])).fit(cov_type='HC1')
m3 = sm.OLS(y, sm.add_constant(df_reg[m3_feats])).fit(cov_type='HC1')

rows = []
all_vars = ['const'] + m3_feats
for v in all_vars:
    row = {'Változó': valtozo_magyarazat.get(v, v)}
    for name, m in [('Modell 1 (Fizikai)', m1), ('Modell 2 (+Elérhetőség)', m2), ('Modell 3 (+Zaj externália)', m3)]:
        if v in m.params:
            sig = '***' if m.pvalues[v]<0.01 else ('**' if m.pvalues[v]<0.05 else ('*' if m.pvalues[v]<0.1 else ''))
            row[name] = f"{m.params[v]:.5f}{sig} (se: {m.bse[v]:.5f})"
        else:
            row[name] = '-'
    rows.append(row)

stat_rows = [
    {'Változó': 'R²', 'Modell 1 (Fizikai)': f'{m1.rsquared:.4f}', 'Modell 2 (+Elérhetőség)': f'{m2.rsquared:.4f}', 'Modell 3 (+Zaj externália)': f'{m3.rsquared:.4f}'},
    {'Változó': 'Korrigált R²', 'Modell 1 (Fizikai)': f'{m1.rsquared_adj:.4f}', 'Modell 2 (+Elérhetőség)': f'{m2.rsquared_adj:.4f}', 'Modell 3 (+Zaj externália)': f'{m3.rsquared_adj:.4f}'},
    {'Változó': 'F-próba p-érték', 'Modell 1 (Fizikai)': f'{m1.f_pvalue:.2e}', 'Modell 2 (+Elérhetőség)': f'{m2.f_pvalue:.2e}', 'Modell 3 (+Zaj externália)': f'{m3.f_pvalue:.2e}'},
    {'Változó': 'Mintaelemszám (N)', 'Modell 1 (Fizikai)': str(int(m1.nobs)), 'Modell 2 (+Elérhetőség)': str(int(m2.nobs)), 'Modell 3 (+Zaj externália)': str(int(m3.nobs))}
]

cmp_df = pd.DataFrame(rows + stat_rows)
cmp_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + cmp_df.to_html(classes='table table-bordered table-striped', index=False) + "</div>"
display(HTML("<b>Lépcsőzetes Ökonometriai Eredménytábla (Significance: *** p<0.01, ** p<0.05, * p<0.1):</b>" + cmp_html))

# Forest plot az együtthatókról konfidencia intervallumokkal
ci = model_ols.conf_int()
ci_df = pd.DataFrame({
    'Valtozo': [valtozo_magyarazat.get(c, c) for c in features],
    'Beta': model_ols.params[features],
    'CI_low': ci.loc[features, 0],
    'CI_high': ci.loc[features, 1]
})

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=ci_df['Beta'],
    y=ci_df['Valtozo'],
    mode='markers',
    error_x=dict(type='data', symmetric=False, array=ci_df['CI_high'] - ci_df['Beta'], arrayminus=ci_df['Beta'] - ci_df['CI_low']),
    marker=dict(color='#2563eb', size=10),
    name='Kettős TOD Modell (95% CI)'
))
fig2.add_vline(x=0, line_dash='dash', line_color='red')
fig2.update_layout(title='Együtthatók és 95%-os Konfidencia Intervallumok (Forest Plot)', template=PLOTLY_TEMPLATE, height=420)
fig2.show()

### 4. Interaktív Hedonikus Modellező Laboratórium

**📌 Próbálja ki saját modell-specifikációit!**
Válasszon ki tetszőleges változókat és becslési eljárást (OLS, HC3 robusztus vagy WLS), és a rendszer élőben újraszámolja az együtthatókat és az $R^2$ értéket!

In [ ]:
# Interaktív hedonikus kalkulátor — a TÉNYLEGES becsült együtthatókkal (HC1 modell)
import json as _json
_coef = {k: float(v) for k, v in model_ols.params.items()}
_median_nm = float(df_reg['nm_ar_huf'].median())
_median_rail = int(df_reg['tavolsag_vasut_m'].median())
_median_railnet = int(df_reg['tavolsag_vasut_halozati_m'].median())
_median_metro = int(df_reg['tavolsag_metro_halozati_m'].median())
_n_obs = int(model_ols.nobs)

_html_tpl = '''<div id="hedonic_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <h3 style="margin:0 0 4px 0; color:#1e3a8a;">Interaktiv hedonikus ertekbecslo (a becsult modell egyutthatoival)</h3>
  <p style="margin:0 0 14px 0; color:#64748b; font-size:13px;">A csuszkak a modell tenyleges beta-i alapjan szamolnak: ln(ar/m2) = b0 + SUM(bi*xi) (HC1 becsles, garantalt pontos elado minta).</p>
  <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:16px;">
    <div style="background:#f8fafc; padding:14px; border-radius:8px;">
      <div style="font-size:12px; font-weight:700; color:#334155; margin-bottom:6px;">Alapterulet: <span id="hed_lbl_area" style="color:#2563eb;">55 m2</span></div>
      <input type="range" id="hed_area" min="30" max="120" value="55" style="width:100%;" oninput="recalcHedonic()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Szobaszam: <span id="hed_lbl_rooms" style="color:#2563eb;">2</span></div>
      <input type="range" id="hed_rooms" min="1" max="5" value="2" step="0.5" style="width:100%;" oninput="recalcHedonic()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Allapot (1-6): <span id="hed_lbl_cond" style="color:#2563eb;">4</span></div>
      <input type="range" id="hed_cond" min="1" max="6" value="4" style="width:100%;" oninput="recalcHedonic()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Emelet: <span id="hed_lbl_floor" style="color:#2563eb;">2</span></div>
      <input type="range" id="hed_floor" min="0" max="10" value="2" style="width:100%;" oninput="recalcHedonic()">
    </div>
    <div style="background:#f8fafc; padding:14px; border-radius:8px;">
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="hed_panel" onchange="recalcHedonic()"> Panel szerkezet</label>
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="hed_lift" onchange="recalcHedonic()"> Lift</label>
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="hed_balcony" checked onchange="recalcHedonic()"> Erkely</label>
      <div style="font-size:12px; font-weight:700; color:#334155; margin:12px 0 6px;">Vaganytol mert legvonal (m): <span id="hed_lbl_rail" style="color:#dc2626;">__MEDIAN_RAIL__</span></div>
      <input type="range" id="hed_rail" min="10" max="2000" value="__MEDIAN_RAIL__" style="width:100%;" oninput="recalcHedonic()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Allomas halozati tav (m): <span id="hed_lbl_railnet" style="color:#f59e0b;">__MEDIAN_RAILNET__</span></div>
      <input type="range" id="hed_railnet" min="100" max="2500" value="__MEDIAN_RAILNET__" style="width:100%;" oninput="recalcHedonic()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Metro halozati tav (m): <span id="hed_lbl_metro" style="color:#2563eb;">__MEDIAN_METRO__</span></div>
      <input type="range" id="hed_metro" min="50" max="2500" value="__MEDIAN_METRO__" style="width:100%;" oninput="recalcHedonic()">
    </div>
  </div>
  <div style="margin-top:16px; background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; display:flex; justify-content:space-between; align-items:center;">
    <div>
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Becsult fajlagos ar</div>
      <div id="hed_result" style="font-size:26px; font-weight:800; color:#1d4ed8;">-- Ft/m2</div>
      <div id="hed_vs_median" style="font-size:12px; color:#2563eb;">--</div>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">N=__N_OBS__ &bull; HC1</span>
  </div>
</div>
<script>
(function() {
  const coef = __COEF__;
  const medianNM = __MEDIAN_NM__;
  window.recalcHedonic = function() {
    const area = +document.getElementById('hed_area').value;
    const rooms = +document.getElementById('hed_rooms').value;
    const cond = +document.getElementById('hed_cond').value;
    const floor = +document.getElementById('hed_floor').value;
    const panel = document.getElementById('hed_panel').checked ? 1 : 0;
    const lift = document.getElementById('hed_lift').checked ? 1 : 0;
    const balcony = document.getElementById('hed_balcony').checked ? 1 : 0;
    const rail = +document.getElementById('hed_rail').value;
    const railnet = +document.getElementById('hed_railnet').value;
    const metro = +document.getElementById('hed_metro').value;
    let lnp = coef['const'];
    lnp += coef['korrigalt_alapterulet_nm'] * area;
    lnp += coef['szobaszam_osszes'] * rooms;
    lnp += coef['is_panel'] * panel;
    lnp += coef['has_lift'] * lift;
    lnp += coef['allapot_kod'] * cond;
    lnp += coef['has_erkely'] * balcony;
    lnp += coef['emelet_szam'] * floor;
    lnp += coef['log_tavolsag_vasut_m'] * Math.log(Math.max(rail, 1));
    lnp += coef['tavolsag_vasut_halozati_m'] * railnet;
    lnp += coef['tavolsag_metro_halozati_m'] * metro;
    const price = Math.exp(lnp);
    document.getElementById('hed_lbl_area').innerText = area + ' m2';
    document.getElementById('hed_lbl_rooms').innerText = rooms;
    document.getElementById('hed_lbl_cond').innerText = cond;
    document.getElementById('hed_lbl_floor').innerText = floor;
    document.getElementById('hed_lbl_rail').innerText = rail;
    document.getElementById('hed_lbl_railnet').innerText = railnet;
    document.getElementById('hed_lbl_metro').innerText = metro;
    document.getElementById('hed_result').innerText = Math.round(price).toLocaleString('hu-HU') + ' Ft/m2';
    const diff = (price / medianNM - 1) * 100;
    document.getElementById('hed_vs_median').innerText = 'A mintamedianhoz kepest: ' + (diff >= 0 ? '+' : '') + diff.toFixed(1) + '%';
  };
  setTimeout(recalcHedonic, 50);
})();
</script>'''

html_hedonic = (_html_tpl.replace('__COEF__', _json.dumps(_coef))
                         .replace('__MEDIAN_NM__', str(_median_nm))
                         .replace('__MEDIAN_RAIL__', str(_median_rail))
                         .replace('__MEDIAN_RAILNET__', str(_median_railnet))
                         .replace('__MEDIAN_METRO__', str(_median_metro))
                         .replace('__N_OBS__', str(_n_obs)))
display(HTML(html_hedonic))
